# LangChain Hands-on 10 — Retrievers
**Trainer:** Ajeetkumar

**What we will practise**
- Vector store → retriever
- similarity, MMR, score threshold
- BM25 keyword retriever
- EnsembleRetriever (hybrid search)
- MultiQueryRetriever (LLM rewrites the question)
- ContextualCompressionRetriever
- ParentDocumentRetriever

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-community langchain-classic langchain-huggingface sentence-transformers faiss-cpu rank_bm25

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# What is a Retriever?
- A retriever takes a **question** and returns **relevant Documents**.
- Every vector store can become a retriever with `.as_retriever()`.
- Retrievers are **Runnables** → use `.invoke()` and put them in chains.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
texts = [
    "Report a lost card within 3 days for zero liability.",
    "Card-not-present payments above Rs 5,000 need OTP.",
    "Credit card annual fee is Rs 500, waived on spends above Rs 1 lakh.",
    "UPI limit for a new payee is Rs 25,000 in the first 24 hours.",
    "Never share your UPI PIN with anyone.",
    "Error code UPI-U30 means the debit failed at the bank.",
    "Video KYC is available from 9 AM to 9 PM.",
    "Home loan interest starts at 8.5% per year.",
]
docs = [Document(page_content=t, metadata={"id": i}) for i, t in enumerate(texts)]
vector_store = FAISS.from_documents(docs, embeddings)

## 1. Basic retriever (similarity, top-k)

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever.invoke("my card is stolen")

## 2. MMR — diverse results

In [ ]:
mmr_retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 8})
mmr_retriever.invoke("card charges and rules")

## 3. Score threshold — only return good matches

In [ ]:
threshold_retriever = vector_store.as_retriever(search_type="similarity_score_threshold",
                                                search_kwargs={"score_threshold": 0.4})
threshold_retriever.invoke("pizza recipe")      # unrelated → may return nothing

## 4. BM25Retriever — keyword search (exact words, codes, IDs)

In [ ]:
from langchain_community.retrievers import BM25Retriever

In [ ]:
bm25 = BM25Retriever.from_documents(docs, k=2)
bm25.invoke("U30")

In [ ]:
retriever.invoke("U30")       # vector search may miss exact codes

## 5. EnsembleRetriever — hybrid = keyword + semantic ⭐

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

In [ ]:
hybrid = EnsembleRetriever(retrievers=[bm25, retriever], weights=[0.5, 0.5])
hybrid.invoke("what does U30 error mean for my UPI payment?")

## 6. MultiQueryRetriever — LLM writes extra versions of your question

In [ ]:
from langchain_classic.retrievers import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)   # to see generated questions

In [ ]:
multi = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)
multi.invoke("how much can I send to someone new?")

## 7. ContextualCompressionRetriever — keep only the useful part of each chunk

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
long_docs = [Document(page_content="NovaBank was founded in 1990 in Pune. It has 500 branches. "
                      "For lost cards, customers must call 1800-111-222 or block the card in the app within 3 days. "
                      "The bank also sponsors a cricket team.")]
long_retriever = FAISS.from_documents(long_docs, embeddings).as_retriever()

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=long_retriever)

In [ ]:
compression_retriever.invoke("how to block a lost card?")

## 8. ParentDocumentRetriever — search small chunks, return the big parent

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
big_docs = [Document(page_content=" ".join(texts[:3]), metadata={"section": "cards"}),
            Document(page_content=" ".join(texts[3:6]), metadata={"section": "upi"})]

In [ ]:
parent_retriever = ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embeddings),
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0),
)
parent_retriever.add_documents(big_docs)

In [ ]:
parent_retriever.invoke("annual fee")      # returns the whole "cards" section

# Which retriever when?
| Retriever | Use when |
|---|---|
| similarity | default |
| MMR | results are repetitive |
| score threshold | avoid unrelated answers |
| BM25 | exact words: IDs, error codes, names |
| **Ensemble (hybrid)** | production RAG — best of both |
| MultiQuery | users ask short / vague questions |
| Contextual compression | chunks are long and noisy |
| ParentDocument | need precise search + full context |

## 🧪 Practice
1. Build a hybrid retriever on a PDF and compare with pure vector search on 5 questions.
2. Print the questions MultiQueryRetriever generates for "cheap phone with good camera".
3. Try different `weights` in EnsembleRetriever.